In [ ]:
# --- Limiter BLAS à 1 thread par worker (évite N_JOBS × BLAS_THREADS) ---
import os
# BLAS threads set after N_JOBS is defined below





# --- Configuration ---
import os, sys, time, re, warnings, gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
%matplotlib inline

from rakuten.common.data_loader import prepare_hierarchical_data_from_splits
from rakuten.common.metrics import (compute_all_metrics, compute_per_class_f1,
                             plot_confusion_matrix, plot_per_class_f1)
from rakuten.common.utils import set_seed

SEED = 42
N_JOBS = int(os.environ.get("N_JOBS", 26))

# BLAS parallélise en C (threads natifs, mémoire partagée, pas de fork)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"  # Défaut 26, surchargeable via env
set_seed(SEED)

PROJECT_ROOT = Path.cwd().parent.parent
DATASET_DIR = Path("/home/infres/tandriam-25/fil_rouge/dataset")

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})


from joblib import Parallel, delayed
import joblib
print(f"Configuration terminée. N_JOBS={N_JOBS}, backend=threading")

In [ ]:
# --- Chargement des données (avec cache) ---
import joblib as _jl

_cache_path = PROJECT_ROOT / "models" / "baselines" / "_data_cache.pkl"
_cache_path.parent.mkdir(parents=True, exist_ok=True)

if _cache_path.exists():
    print("Chargement depuis le cache...")
    t0 = time.time()
    data = _jl.load(_cache_path)
    print(f"Cache chargé en {time.time() - t0:.0f}s")
else:
    print("Chargement des splits train/val/test...")
    t0 = time.time()
    data = prepare_hierarchical_data_from_splits(
        train_path=str(DATASET_DIR / "train_split.jsonl"),
        val_path=str(DATASET_DIR / "val_split.jsonl"),
        test_path=str(DATASET_DIR / "test_split.jsonl"),
        n_samples=None,
        seed=SEED,
    )
    _jl.dump(data, _cache_path, compress=3)
    print(f"Chargement terminé en {time.time() - t0:.0f}s — cache sauvegardé")

print(f"  Train : {len(data['X_train']):,} | Val : {len(data['X_val']):,} | Test : {len(data['X_test']):,}")
print(f"  L1 : {data['num_classes_l1']} | L2 : {data['num_classes_l2']}")

<a id="section-2"></a>
## 2. Entraînement Word2Vec (Skip-gram)

On entraîne un modèle Word2Vec Skip-gram sur le corpus d'entraînement.
Chaque mot est représenté par un vecteur dense de 200 dimensions.


In [ ]:
# --- Stopwords & Tokenisation ---
import nltk

try:
    french_stops = set(nltk.corpus.stopwords.words("french"))
    english_stops = set(nltk.corpus.stopwords.words("english"))
except LookupError:
    nltk.download("stopwords", quiet=True)
    french_stops = set(nltk.corpus.stopwords.words("french"))
    english_stops = set(nltk.corpus.stopwords.words("english"))

all_stops = french_stops | english_stops

def tokenize(text):
    """Tokenise un texte : lowercase, suppression stopwords et tokens courts."""
    tokens = re.findall(r"\b\w+\b", str(text).lower())
    return [t for t in tokens if len(t) >= 3 and t not in all_stops]

print("Tokenisation du corpus d'entraînement...")
t0 = time.time()
train_tokens = [tokenize(text) for text in data["X_train"]]
print(f"Tokenisation terminée en {time.time() - t0:.0f}s")
print(f"  Documents : {len(train_tokens):,}")
print(f"  Tokens moyen/doc : {np.mean([len(t) for t in train_tokens]):.0f}")

In [ ]:
# --- Entraînement Word2Vec ---
from gensim.models import Word2Vec
import joblib
_save_dir = PROJECT_ROOT / "models" / "baselines" / "word2vec"
_save_dir.mkdir(parents=True, exist_ok=True)
_w2v_path = _save_dir / "word2vec_model.bin"

if _w2v_path.exists():
    w2v_model = Word2Vec.load(str(_w2v_path))
    print(f"Word2Vec chargé depuis {_w2v_path} ({len(w2v_model.wv)} mots)")
else:
    print(f"Entraînement Word2Vec (skip-gram, 200d, {N_JOBS} threads)...")
    t0 = time.time()
    w2v_model = Word2Vec(
        sentences=train_tokens, vector_size=200, window=5,
        min_count=3, sg=1, epochs=10, workers=N_JOBS,
    )
    w2v_model.save(str(_w2v_path))
    print(f"Word2Vec entraîné en {time.time()-t0:.0f}s — {len(w2v_model.wv)} mots — sauvegardé")

print(f"  Vocabulaire : {len(w2v_model.wv):,} mots")
print(f"  Dimensions : {w2v_model.wv.vector_size}")

<a id="section-3"></a>
## 3. Vectorisation des documents

Chaque document est représenté par la **moyenne des embeddings** de ses tokens.
Les tokens absents du vocabulaire Word2Vec sont ignorés.


In [ ]:
# --- Vectorisation par moyenne des embeddings (parallèle, threads) ---
# Threads = même mémoire, pas de fork → pas de duplication RAM
# (loky/fork dupliquait ~20 Go × 42 workers = 840 Go à cause du refcount CPython)

_vectors_path = PROJECT_ROOT / "models" / "baselines" / "word2vec" / "vectors.npy"
np.save(str(_vectors_path), w2v_model.wv.vectors)
_vectors_mmap = np.load(str(_vectors_path), mmap_mode='r')
_key_to_index = w2v_model.wv.key_to_index
_vector_size = w2v_model.wv.vector_size

def _doc2vec_chunk(tokens_chunk):
    """Vectorise un chunk — accède aux variables notebook (threads = mémoire partagée)."""
    result = np.zeros((len(tokens_chunk), _vector_size), dtype=np.float32)
    for i, tokens in enumerate(tokens_chunk):
        idxs = [_key_to_index[t] for t in tokens if t in _key_to_index]
        if idxs:
            result[i] = _vectors_mmap[idxs].mean(axis=0)
    return result

def vectorize_tokens(tokens_list, n_jobs):
    """Vectorise des tokens pré-calculés en parallèle (threads, RAM partagée)."""
    chunk_size = max(1, len(tokens_list) // n_jobs)
    chunks = [tokens_list[i:i+chunk_size] for i in range(0, len(tokens_list), chunk_size)]
    results = Parallel(n_jobs=n_jobs, backend='threading')(
        delayed(_doc2vec_chunk)(chunk) for chunk in chunks
    )
    return np.vstack(results)

# Train : réutilise train_tokens de cell 3 (pas de re-tokenisation)
print(f"Vectorisation train ({N_JOBS} threads, RAM partagée)...")
t0 = time.time()
X_train_w2v = vectorize_tokens(train_tokens, N_JOBS)
del train_tokens; gc.collect()
print(f"  Train : {X_train_w2v.shape} en {time.time()-t0:.0f}s")

# Val : tokenise puis vectorise
print("Vectorisation val...")
t0 = time.time()
val_tokens = [tokenize(t) for t in data["X_val"]]
X_val_w2v = vectorize_tokens(val_tokens, N_JOBS)
del val_tokens; gc.collect()
print(f"  Val : {X_val_w2v.shape} en {time.time()-t0:.0f}s")

# Test : tokenise puis vectorise
print("Vectorisation test...")
t0 = time.time()
test_tokens = [tokenize(t) for t in data["X_test"]]
X_test_w2v = vectorize_tokens(test_tokens, N_JOBS)
del test_tokens; gc.collect()
print(f"  Test : {X_test_w2v.shape} en {time.time()-t0:.0f}s")

# Libérer les textes bruts (on garde seulement vecteurs + labels)
del data["X_train"], data["X_val"], data["X_test"]
gc.collect()
print("Textes bruts libérés de la mémoire.")

<a id="section-4"></a>
## 4. Classification L1 (157 catégories)


In [ ]:
# --- Entraînement L1 ---
# LinearSVC mono-thread (dense matrix = fork duplique la RAM)
# Modèle sauvegardé → re-exécution instantanée
_clf_l1_path = PROJECT_ROOT / "models" / "baselines" / "word2vec" / "clf_l1.joblib"

if _clf_l1_path.exists():
    clf_l1 = joblib.load(_clf_l1_path)
    print(f"Classifieur L1 chargé depuis {_clf_l1_path}")
    train_time_l1 = 0
else:
    from sklearn.svm import LinearSVC
    print("Entraînement L1 : LinearSVC (157 classes, mono-thread)...")
    t0 = time.time()
    clf_l1 = LinearSVC(C=1.0, class_weight="balanced", max_iter=2000, dual="auto")
    clf_l1.fit(X_train_w2v, data["y_l1_train"])
    train_time_l1 = time.time() - t0
    joblib.dump(clf_l1, _clf_l1_path)
    print(f"L1 entraîné en {train_time_l1:.0f}s — sauvegardé")

print("Prédiction L1...")
t0 = time.time()
preds_l1_train = clf_l1.predict(X_train_w2v)
preds_l1_val = clf_l1.predict(X_val_w2v)
preds_l1_test = clf_l1.predict(X_test_w2v)
print(f"Prédiction L1 terminée en {time.time()-t0:.0f}s")

<a id="section-5"></a>
## 5. Classification L2 (6000+ sous-catégories)


In [ ]:
# --- Entraînement L2 ---
# SGDClassifier mono-thread (dense matrix = fork duplique la RAM)
# Modèle sauvegardé → re-exécution instantanée
_clf_l2_path = PROJECT_ROOT / "models" / "baselines" / "word2vec" / "clf_l2.joblib"

if _clf_l2_path.exists():
    clf_l2 = joblib.load(_clf_l2_path)
    print(f"Classifieur L2 chargé depuis {_clf_l2_path}")
    train_time_l2 = 0
else:
    from sklearn.linear_model import SGDClassifier
    print("Entraînement L2 : SGDClassifier (6126 classes, mono-thread)...")
    t0 = time.time()
    clf_l2 = SGDClassifier(
        loss="hinge", class_weight="balanced",
        alpha=1e-4, max_iter=100, n_jobs=1,
    )
    clf_l2.fit(X_train_w2v, data["y_l2_train"])
    train_time_l2 = time.time() - t0
    joblib.dump(clf_l2, _clf_l2_path)
    print(f"L2 entraîné en {train_time_l2:.0f}s — sauvegardé")

print("Prédiction L2...")
t0 = time.time()
preds_l2_test = clf_l2.predict(X_test_w2v)
print(f"Prédiction L2 terminée en {time.time()-t0:.0f}s")

<a id="section-6"></a>
## 6. Métriques hiérarchiques


In [ ]:
# --- Métriques hiérarchiques ---
all_metrics = compute_all_metrics(
    labels_l1=data["y_l1_test"], preds_l1=preds_l1_test,
    labels_l2=data["y_l2_test"], preds_l2=preds_l2_test,
    l2_to_l1_mapping=data["l2_to_l1_mapping"],
)

print("=== Métriques Word2Vec + LinearSVC/SGD (Test Set) ===")
for k, v in all_metrics.items():
    if isinstance(v, float):
        print(f"  {k:30s} : {v:.4f}")

summary = pd.DataFrame([{
    "Modèle": "Word2Vec + LinearSVC/SGD",
    "F1 L1 macro": all_metrics["f1_l1_macro"],
    "F1 L2 macro": all_metrics["f1_l2_macro"],
    "HCR": all_metrics.get("hcr", 0.0),
}])
display(summary.style.format({
    "F1 L1 macro": "{:.4f}", "F1 L2 macro": "{:.4f}", "HCR": "{:.4f}",
}).set_caption("Word2Vec Baseline — dataset complet 5.5M"))

<a id="section-7"></a>
## 7. Visualisations


In [ ]:
# --- Matrices de confusion & F1 par classe ---
per_class_f1_l1 = compute_per_class_f1(data["y_l1_test"], preds_l1_test, data["l1_encoder"])

fig_cm = plot_confusion_matrix(
    data["y_l1_test"], preds_l1_test, data["l1_encoder"],
    title="Matrice de confusion L1 — Word2Vec Baseline", top_n=30,
)
plt.show()

fig_f1 = plot_per_class_f1(per_class_f1_l1, title="F1 par classe L1 — Word2Vec", top_n=20)
plt.show()


In [ ]:
# --- F1 par classe L2 ---
per_class_f1_l2 = compute_per_class_f1(data["y_l2_test"], preds_l2_test, data["l2_encoder"])
fig_f1_l2 = plot_per_class_f1(per_class_f1_l2, title="F1 par classe L2 — Word2Vec", top_n=20)
plt.show()

f1_vals = list(per_class_f1_l2.values())
print(f"F1 L2 — Moyenne: {np.mean(f1_vals):.4f}, Médiane: {np.median(f1_vals):.4f}")
print(f"  Classes avec F1 = 0 : {sum(1 for v in f1_vals if v == 0)}")
print(f"  Classes avec F1 > 0.5 : {sum(1 for v in f1_vals if v > 0.5)}")


In [ ]:
# --- Sauvegarde per-class F1 en JSON (pour la comparaison) ---
import json as _json
_out_dir = PROJECT_ROOT / "results" / "baselines"
_out_dir.mkdir(parents=True, exist_ok=True)

_l1_path = _out_dir / "per_class_f1_l1_word2vec.json"
_l2_path = _out_dir / "per_class_f1_l2_word2vec.json"
_l1_path.write_text(_json.dumps(per_class_f1_l1, ensure_ascii=False, indent=2))
_l2_path.write_text(_json.dumps(per_class_f1_l2, ensure_ascii=False, indent=2))
print(f"Per-class F1 sauvegardé :")
print(f"  L1: {_l1_path} ({len(per_class_f1_l1)} classes)")
print(f"  L2: {_l2_path} ({len(per_class_f1_l2)} classes)")


<a id="section-9"></a>
## 9. Conclusion

### Word2Vec vs TF-IDF
- Word2Vec capture des **relations sémantiques** (synonymes, analogies) que TF-IDF ignore
- Mais la moyenne des embeddings perd l'**information d'ordre** et de **fréquence** des termes
- TF-IDF reste généralement supérieur pour la classification de texte avec un classifieur linéaire

### Ce que HYDRA apporte
- **Embeddings contextuels** : CamemBERT produit des représentations dépendantes du contexte
- **Fine-tuning end-to-end** : l'encodeur s'adapte à la tâche de classification
- **Architecture hiérarchique** : modélisation explicite de la relation L1→L2

---
*Notebook généré dans le cadre du projet de classification hiérarchique Rakuten — Télécom Paris x Rakuten France, 2026.*
